<a href="https://colab.research.google.com/github/joseguzman1337/aws-financial-ai-agent/blob/main/invocation_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AWS Financial AI Agent - Invocation Demo

This notebook demonstrates end-to-end user authentication, live agent invocation, and observability trace extraction **without requiring any AWS access keys** from the recruiter. Credentials and secrets are retrieved securely via Cognito Identity Pools and AWS SSM.

In [41]:
# Install dependencies silently
!pip install boto3 requests -q
!sudo apt-get update -y -q > /dev/null && sudo apt-get install neofetch -y -q > /dev/null
!neofetch

import json
import time
import urllib.parse
import uuid

import boto3
import requests
from botocore.auth import SigV4Auth
from botocore.awsrequest import AWSRequest
from botocore.session import Session

# --- Global config / aliases ---
CFG = {
    "region": "us-east-1",
    "agent_arn": "arn:aws:bedrock-agentcore:us-east-1:162187491349:runtime/Financial_Analyst_Agent-hvRgckAqaW",
}

PARAMS = {
    "langfuse_pk": "/financial-ai/langfuse/public-key",
    "langfuse_sk": "/financial-ai/langfuse/secret-key",
}

ENCODED_ARN = urllib.parse.quote(CFG["agent_arn"], safe="")
AGENTCORE_URL = (
    f"https://bedrock-agentcore.{CFG['region']}.amazonaws.com"
    f"/runtimes/{ENCODED_ARN}/invocations"
)
MASTER_SESSION_ID = str(uuid.uuid4())

# Direct AWS clients (passwordless)
sts = boto3.client("sts", region_name=CFG["region"])
ssm = boto3.client("ssm", region_name=CFG["region"])


def ssm_get(name: str) -> str:
    return ssm.get_parameter(Name=name, WithDecryption=True)["Parameter"]["Value"]


def _sigv4_headers(url: str, body: bytes, session_id: str) -> dict:
    aws_session = Session()
    creds = aws_session.get_credentials()
    if creds is None:
        raise RuntimeError("No AWS credentials found. Configure AWS auth in notebook runtime.")

    frozen = creds.get_frozen_credentials()
    headers = {
        "Content-Type": "application/json",
        "Accept": "text/event-stream",
        "X-Amzn-Bedrock-AgentCore-Runtime-Session-Id": session_id,
    }
    req = AWSRequest(method="POST", url=url, data=body, headers=headers)
    SigV4Auth(frozen, "bedrock-agentcore", CFG["region"]).add_auth(req)
    return dict(req.prepare().headers)


def query_financial_agent(prompt: str):
    payload = json.dumps({"prompt": prompt}).encode("utf-8")
    headers = _sigv4_headers(AGENTCORE_URL, payload, MASTER_SESSION_ID)

    print(f"\n--- Query: {prompt} ---")
    resp = requests.post(
        AGENTCORE_URL,
        headers=headers,
        data=payload,
        stream=True,
        timeout=120,
    )
    if resp.status_code != 200:
        print(f"Error {resp.status_code}: {resp.text[:400]}")
        return

    for line in resp.iter_lines():
        if not line:
            continue
        decoded = line.decode("utf-8")
        if decoded.startswith("data: "):
            data = json.loads(decoded[6:])
            if "event" in data:
                event = data["event"]
                if isinstance(event, str):
                    print(event)
                elif isinstance(event, list):
                    for part in event:
                        if isinstance(part, dict) and part.get("type") == "text":
                            print(part.get("text", ""), end="")
                    print()
                else:
                    print(event)
            elif "error" in data:
                print(f"Agent error: {data['error'][:300]}")


print(f"Session ID: {MASTER_SESSION_ID}")
print("Mode: Direct AWS SigV4 (passwordless)")


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
            .-/+oossssoo+/-.
        `:+ssssssssssssssssss+:`
      -+ssssssssssssssssssyyssss+-
    .ossssssssssssssssssdMMMNysssso.
   /ssssssssssshdmmNNmmyNMMMMhssssss/
  +ssssssssshmydMMMMMMMNddddyssssssss+
 /sssssssshNMMMyhhyyyyhmNMMMNhssssssss/
.ssssssssdMMMNhsssssssssshNMMMdssssssss.
+sssshhhyNMMNyssssssssssssyNMMMysssssss+
ossyNMMMNyMMhsssssssssssssshmmmhssssssso
ossyNMMMNyMMhsssssssssssssshmmmhssssssso
+sssshhhyNMMNyssssssssssssyNMMMysssssss+
.ssssssssdMMMNhsssssssssshNMMMdssssssss.
 /sssssssshNMMMyhhyyyyhdNMMMNhssssssss/
  +sssssssssdmydMMMMMMMMddddyssssssss+
   /ssssssssssshdmNNNNmyNMMMMhssssss/
    .ossssssssssssssssssdMMMNysssso.
      -+sssssssssssssssssyyyssss+-
        `:+ssssssssssssssssss+:`
            .-/+oossssoo+/-.
root@d768b65d2734 
----------------- 
OS: Ubuntu 22.04.5 LTS x8

### 1. Retrieve Guest Login Credentials
We use the Cognito Identity Pool (Unauthenticated) to securely fetch the login credentials from AWS SSM.

In [ ]:
try:
    caller = sts.get_caller_identity()
    print(f"Success: AWS identity detected ({caller['Arn']}).")
except Exception as e:
    print(f"Error: AWS credentials unavailable: {str(e)}")


### 2. Authenticate with Amazon Cognito
Login using the credentials retrieved in the previous step.

In [43]:
print("No Cognito password flow required in notebook.")
print("Agent calls use direct AWS SigV4 signing.")


Success: Cognito Authentication Successful.


### 3. Invoke the Agentcore Streaming Endpoint

In [ ]:
print(f"Agent URL configured: {AGENTCORE_URL}")
print(f"Session ID: {MASTER_SESSION_ID}")


### 4. Execute Required Financial Queries

In [45]:
queries = [
    "What is the stock price for Amazon right now?",
    "What were the stock prices for Amazon in Q4 last year?",
    "Compare Amazon's recent stock performance to what analysts predicted in their reports.",
    "I’m researching AMZN give me the current price and any relevant information about their AI business.",
    "What is the total amount of office space Amazon owned in North America in 2024?"
]
for q in queries: query_financial_agent(q)


--- Query: What is the stock price for Amazon right now? ---
Error 424: {"message":"An error occurred when starting the runtime. Please check your CloudWatch logs for more information."}

--- Query: What were the stock prices for Amazon in Q4 last year? ---
Error 424: {"message":"An error occurred when starting the runtime. Please check your CloudWatch logs for more information."}

--- Query: Compare Amazon's recent stock performance to what analysts predicted in their reports. ---
Error 424: {"message":"An error occurred when starting the runtime. Please check your CloudWatch logs for more information."}

--- Query: I’m researching AMZN give me the current price and any relevant information about their AI business. ---
Error 424: {"message":"An error occurred when starting the runtime. Please check your CloudWatch logs for more information."}

--- Query: What is the total amount of office space Amazon owned in North America in 2024? ---
Error 424: {"message":"An error occurred when s

### 5. Observability Verification (Authenticated AWS Identity)
Fetch Langfuse traces securely using temporary credentials obtained via the Cognito session.

**Session ID:** `e9ab4997-f01b-41ca-abde-bfb7cf06c258`

In [ ]:
try:
    pk = ssm_get(PARAMS["langfuse_pk"])
    sk = ssm_get(PARAMS["langfuse_sk"])
    print(f"Success: retrieved Langfuse keys (PK: {pk[:7]}...)")

    if "placeholder" in sk.lower() or "00000000" in sk:
        print("NOTE: Langfuse keys are placeholders. Traces will not be available until real keys are stored in SSM.")
    else:
        print("Waiting 10s for trace propagation...")
        time.sleep(10)

        found = False
        for host in [
            "https://us.cloud.langfuse.com",
            "https://cloud.langfuse.com",
            "https://eu.cloud.langfuse.com",
        ]:
            trace_url = f"{host}/api/public/sessions/{MASTER_SESSION_ID}"
            trace_resp = requests.get(trace_url, auth=(pk, sk), timeout=30)
            if trace_resp.status_code == 200:
                print(f"\n--- Langfuse Traces (Host: {host}, Session: {MASTER_SESSION_ID}) ---")
                print(json.dumps(trace_resp.json(), indent=2)[:2000])
                found = True
                break

        if not found:
            print(f"No traces found for session {MASTER_SESSION_ID}. Check Langfuse dashboard manually.")
except Exception as e:
    print(f"Error: Observability retrieval failed: {str(e)}")
